# SIDIX — fine-tune Qwen2.5-7B-Instruct (QLoRA) di Kaggle

Versi kurasi repo (tanpa output besar). Sumber: salinan lokal sidix-gen.ipynb.

**Perbaikan:** (1) ChatML end-token di berkas Downloads korup; sel kode memakai im_start/im_end dibangun dari string. (2) Sel zip: gunakan !cd ... bukan cd ... di sel Python.

Dataset: /kaggle/input/datasets/mighan/sidix-sft-dataset/finetune_sft.jsonl  
Output: /kaggle/working/sidix-lora-adapter/ lalu zip untuk unduh.


In [ ]:
!pip install -q peft trl bitsandbytes accelerate datasets

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

DATASET_PATH = "/kaggle/input/datasets/mighan/sidix-sft-dataset/finetune_sft.jsonl"
OUTPUT_DIR = "/kaggle/working/sidix-qwen2.5-7b-lora"
ADAPTER_DIR = "/kaggle/working/sidix-lora-adapter"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
dataset = Dataset.from_list(records)
print(f"Dataset loaded: {len(dataset)} samples")

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
print("Model loaded OK")

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


def format_chatml(example):
    im_start = "<|im_" + "start|>"
    im_end = "<|im_" + "end|>"
    text = ""
    for msg in example["messages"]:
        text += f"{im_start}{msg['role']}\n{msg['content']}{im_end}\n"
    text += f"{im_start}assistant\n"
    return {"text": text}


train_dataset = train_dataset.map(format_chatml)
eval_dataset = eval_dataset.map(format_chatml)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
!cd /kaggle/working && zip -r sidix-lora-adapter.zip sidix-lora-adapter